# Foldseek SCOP40 all-vs-all: homologs found at 0.01, E-value vs SG p-value

Every ASTRAL40 domain (SCOPe 2.08, 15,177 structures) is searched against the
non-redundant SCOP40 Foldseek database with **one** `src/foldseek.py` call per
alignment type. The wrapper takes the whole query database and does the rest:
- it makes 50 seeded decoys for every query;
- it searches the decoys and the real queries;
- it gives every hit an SG p-value.

SCOP labels provide the ground truth. A hit is a **homolog** if query and target share a
superfamily or family, and a **non-homolog** if they come from different folds. Same-fold,
different-superfamily pairs are uncertain and not counted. At the cutoff 0.01, we count how
many homologs and non-homologs Foldseek's E-value and the SG p-value each report.

| Step | What | Output |
|---|---|---|
| 1 | non-redundant target `scop40_unique` | `data/scop40_run/foldseek/db/` |
| 2 | query database: first model of every ASTRAL40 structure | `results/scop40_run/foldseek/query_db/` |
| 3 | `foldseek.py` on the query database, alignment types 0 and 2 | `results/scop40_run/foldseek/all_vs_all_aln<T>_sgpval.tsv[.gz]` |
| 4 | homologs and non-homologs found at 0.01 | table |

**Before you start:** run this notebook from `tutorial/` or the repository root. From
Zenodo, unpack these at the repository root:
- `results_scop40_run.tar.gz` holds the precomputed search results. With it, only step 4
  runs, in a few minutes.
- To search from scratch, you also need `data_scop40_run.tar.gz` (structures, labels and
  Foldseek target database in `data/scop40_run/`) and the tools from `bash scripts/build.sh`. Each
  alignment type takes about 7–8 h on 32 threads.

With `USE_CACHED = True` (the default), a step is skipped when its output exists. The
table after the parameters shows what will run.

## Parameters

In [1]:
from pathlib import Path
import os
if not Path('scripts/build.sh').exists():   # opened from tutorial/: work from the repository root
    os.chdir('..')
ROOT = Path('.')                            # all paths below are relative to the repository root

FOLDSEEK  = ROOT / 'bin/foldseek'
DATA_DIR  = ROOT / 'data/scop40_run'        # inputs (Zenodo data_scop40_run.tar.gz)
RUN_DIR   = ROOT / 'results/scop40_run/foldseek'   # results (Zenodo results_scop40_run.tar.gz)
PDBSTYLE  = DATA_DIR / 'pdbstyle-2.08'
TARGET_DB = DATA_DIR / 'foldseek/db/scop40_unique'
FASTA     = DATA_DIR / 'astral.fa'          # ASTRAL40 ids with SCOP labels
LOOKUP    = ROOT / 'data/lookup/m50.txt'

ALN_TYPES  = [0, 2]   # foldseek --alignment-type: 0 = 3Di only, 2 = 3Di+AA (Foldseek's default)
THREADS    = 32
CUTOFF     = 0.01
USE_CACHED = True     # reuse results already in RUN_DIR (e.g. the Zenodo results); False = search again

In [2]:
import re
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, 'src')
from foldseek import get_original_id, get_seq_label_map


def run(cmd, log=None):
    """Run a command, keep its output in `log`, fail loudly."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    with open(log, 'w') if log else open(os.devnull, 'w') as fh:
        subprocess.run(cmd, check=True, stdout=fh, stderr=subprocess.STDOUT)


def result_file(t):
    """The wrapper output for alignment type t (gzipped as shipped on Zenodo, or as written by step 3)."""
    for f in [RUN_DIR / f'all_vs_all_aln{t}_sgpval.tsv.gz', RUN_DIR / f'all_vs_all_aln{t}_sgpval.tsv']:
        if f.exists() and f.stat().st_size > 0:
            return f
    return RUN_DIR / f'all_vs_all_aln{t}_sgpval.tsv'


def search_needed(t):
    return not USE_CACHED or not result_file(t).exists()


pd.DataFrame([{'alignment type': t, 'search results': 'cached' if not search_needed(t) else '-',
               'will run': 'search (~7-8 h) + counting' if search_needed(t) else 'counting only'}
              for t in ALN_TYPES])

,alignment type,search results,will run
0,0,cached,counting only
1,2,cached,counting only


## 1. Target database

`pdbstyle-2.08` holds several entries for some domains (e.g. NMR models). Clustering at
100% sequence identity and 95% coverage keeps one representative each.

In [3]:
if any(search_needed(t) for t in ALN_TYPES) and not Path(f'{TARGET_DB}.dbtype').exists():
    db = TARGET_DB.parent
    db.mkdir(parents=True, exist_ok=True)
    run([FOLDSEEK, 'createdb', PDBSTYLE, db / 'scop40_db', '--threads', THREADS], log=db / 'createdb.log')
    run([FOLDSEEK, 'cluster', db / 'scop40_db', db / 'scop40_clu', db / 'tmp', '-c', 0.95, '--min-seq-id', 1.0,
         '--cov-mode', 0, '--threads', THREADS], log=db / 'cluster.log')
    run([FOLDSEEK, 'createsubdb', db / 'scop40_clu', db / 'scop40_db', TARGET_DB, '--subdb-mode', 1],
        log=db / 'createsubdb.log')
elif Path(f'{TARGET_DB}.dbtype').exists():
    print(f'cached: {TARGET_DB}')
else:
    print('target database not needed (search results cached)')

cached: data/scop40_run/foldseek/db/scop40_unique


## 2. Query database

`createdb` turns every model of an NMR ensemble into its own entry. As in the paper, only
the **first model** of each structure is used as a query. The query set is the ASTRAL40 ids
in `astral.fa`; 49 of them have no usable structure (48 missing from pdbstyle-2.08, 1
unreadable), which leaves 15,128 queries.

In [4]:
QUERY_DB = RUN_DIR / 'query_db/query'

if any(search_needed(t) for t in ALN_TYPES) and not Path(f'{QUERY_DB}.dbtype').exists():
    qdir = QUERY_DB.parent
    qdir.mkdir(parents=True, exist_ok=True)
    run([FOLDSEEK, 'createdb', PDBSTYLE, qdir / 'all_entries', '--threads', THREADS], log=qdir / 'createdb.log')
    astral_ids = {l[1:].split()[0] for l in open(FASTA) if l.startswith('>')}
    # .source: source id -> "<file stem>_MODEL_<n>";  .lookup: key, entry name, source id
    stem = {}
    for line in open(qdir / 'all_entries.source'):
        num, name = line.rstrip('\n').split('\t')
        stem[num] = re.sub(r'_MODEL_\d+$', '', name)
    first = {}
    for line in open(qdir / 'all_entries.lookup'):
        key, name, num = line.rstrip('\n').split('\t')[:3]
        if stem[num] in astral_ids and (stem[num] not in first or int(key) < int(first[stem[num]])):
            first[stem[num]] = key
    (qdir / 'first_entries.txt').write_text(''.join(f'{k}\n' for k in sorted(first.values(), key=int)))
    for s in ['', '_ss', '_h', '_ca']:
        run([FOLDSEEK, 'createsubdb', qdir / 'first_entries.txt', qdir / f'all_entries{s}', f'{QUERY_DB}{s}',
             '--subdb-mode', 0], log=qdir / 'createsubdb.log')
    print(f'{len(first):,} queries -> {QUERY_DB}')
elif Path(f'{QUERY_DB}.dbtype').exists():
    print(f'cached: {QUERY_DB}')
else:
    print('query database not needed (search results cached)')

query database not needed (search results cached)


## 3. Search: one wrapper call per alignment type

The settings are:
- **Decoys:** 50 per query, `--seed 1`.
- **Both searches:** `-s 10.5 --max-seqs 1000 -e 1e9`.
- **Decoy search only:** `--no-rev-align 1 --comp-bias-corr 0 --gap-open 2`.

Because `structureshuffle` restarts its random generator for every query, the result is
identical to searching the queries one at a time. The output has one row per hit:
`query target bits evalue sgpval`. The decoy hits stay in
`results/scop40_run/foldseek/all_vs_all_aln<T>/` (about 35 GB; delete when done).

In [5]:
for t in ALN_TYPES:
    if not search_needed(t):
        print(f'cached: {result_file(t)}')
        continue
    run([sys.executable, 'src/foldseek.py', '--foldseek_url', FOLDSEEK, '--lookup_url', LOOKUP,
         '--num_shuffle', 50, '--seed', 1, 'easy-search',
         '-query', QUERY_DB, '-db', TARGET_DB, '-out', RUN_DIR / f'all_vs_all_aln{t}.tsv',
         '--threads', THREADS, '-s', 10.5, '--max-seqs', 1000, '-e', '1e9', '--alignment-type', t,
         '--shuf-comp-bias-corr', 0, '--shuf-gap-open', 2], log=RUN_DIR / f'wrapper_aln{t}.log')

pd.read_csv(result_file(ALN_TYPES[0]), sep='\t', nrows=5)

cached: results/scop40_run/foldseek/all_vs_all_aln0_sgpval.tsv.gz
cached: results/scop40_run/foldseek/all_vs_all_aln2_sgpval.tsv.gz


,query,target,bits,evalue,sgpval
0,d1k3xa1,d1k3xa1,107.320,6.180000e-09,8.716000e-15
1,d1k3xa1,d1tdza1,36.703,3.170000e-03,8.698000e-10
2,d1k3xa1,d1r2za1,35.042,4.826000e-03,2.546000e-09
3,d1k3xa1,d2i0za2,19.837,1.311000e-01,1.254000e-04
4,d1k3xa1,d2hkja1,19.835,1.031000e-01,1.257000e-04


## 4. Homologs and non-homologs found at 0.01

The counting rules are:
- **Counted:** query–target pairs; self-hits and pairs without SCOP labels are excluded.
- **Homolog:** same superfamily or family.
- **Non-homolog:** different fold.
- **Reported at the cutoff:** a pair is reported when its score is ≤ `CUTOFF`.

These are the counts given in the paper.

In [6]:
labels = get_seq_label_map(FASTA)            # ASTRAL id -> (fold, superfamily, family)


def scop_counts(path):
    df = pd.read_csv(path, sep='\t', usecols=['query', 'target', 'evalue', 'sgpval'])
    df['sgpval'] = pd.to_numeric(df['sgpval'], errors='coerce')      # NA = query without decoy scores
    model = df['query'].str.extract(r'_MODEL_(\d+)')[0].astype(float)
    df = df[~(model > 1).to_numpy()]                                  # only model 1 is a query
    df = df[~df['target'].str.contains('.', regex=False)]             # malformed ids (e.g. d1pvc.1_4)
    norm = {i: get_original_id(i) for i in pd.unique(pd.concat([df['query'], df['target']]))}
    q, tg = df['query'].map(norm), df['target'].map(norm)
    keep = (q.map(labels).notna() & tg.map(labels).notna() & (q != tg)).to_numpy()   # labelled, not self
    df, q, tg = df[keep], q[keep], tg[keep]
    ql, tl = np.array(q.map(labels).tolist()), np.array(tg.map(labels).tolist())
    homolog = ql[:, 1] == tl[:, 1]                                    # same superfamily (or family)
    nonhomolog = ql[:, 0] != tl[:, 0]                                 # different fold
    rows = []
    for metric, name in [('evalue', 'E-value'), ('sgpval', 'SG p-value')]:
        hit = (df[metric] <= CUTOFF).to_numpy()
        rows.append({'score': name, 'homologs found': int((hit & homolog).sum()),
                     'non-homologs found': int((hit & nonhomolog).sum())})
    return rows


table = pd.DataFrame([{'alignment type': t, **r} for t in ALN_TYPES for r in scop_counts(result_file(t))])
table

  0%|          | 0/15177 [00:00<?, ?it/s]

100%|██████████| 15177/15177 [00:00<00:00, 239354.14it/s]

,alignment type,score,homologs found,non-homologs found
0,0,E-value,384473,44273
1,0,SG p-value,443333,17250
2,2,E-value,416819,31650
3,2,SG p-value,492825,30592
